# Sesi 7 — Knowledge ERP dengan LLM Generate (Qwen)

Bagian dari kurikulum **Agentic Knowledge System — FastAPI + DuckDB + Qwen (Local, tanpa LLM cloud)**.

> Jalankan sel dari atas ke bawah secara berurutan. Sel bertanda `🔨 TODO` wajib dilengkapi peserta; solusi contoh disediakan pada sel setelahnya.

## 0. Setup Environment

In [ ]:
!pip -q install llama-cpp-python duckdb pandas

## 1. Model Qwen Lokal
Model yang sama dipakai di seluruh kurikulum (Sesi 3, 6, 4, 7, 8) — cukup diunduh sekali per sesi Colab.

In [ ]:
import os

MODEL_PATH = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_k_m.gguf"

from llama_cpp import Llama
llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_threads=4, verbose=False)

def qwen_llm_call(prompt, stop=None):
    out = llm(prompt, max_tokens=300, stop=stop)
    return out["choices"][0]["text"].strip()

## 2. Siapkan Ulang Data ERP (dari Sesi 5-6) + Seed Transaksi
Agar ada data untuk dilaporkan, kita tambahkan beberapa order contoh.

In [ ]:
import duckdb, uuid

erp_con = duckdb.connect("erp.duckdb")
erp_con.execute("""
CREATE TABLE IF NOT EXISTS products (id VARCHAR PRIMARY KEY, name VARCHAR, price DOUBLE, stock INTEGER);
CREATE TABLE IF NOT EXISTS customers (id VARCHAR PRIMARY KEY, name VARCHAR, email VARCHAR);
CREATE TABLE IF NOT EXISTS orders (id VARCHAR PRIMARY KEY, customer_id VARCHAR, status VARCHAR, created_at TIMESTAMP DEFAULT current_timestamp);
CREATE TABLE IF NOT EXISTS order_items (id VARCHAR PRIMARY KEY, order_id VARCHAR, product_id VARCHAR, qty INTEGER, price DOUBLE);
""")

if erp_con.execute("SELECT COUNT(*) FROM products").fetchone()[0] == 0:
    for name, price, stock in [("Kaos Polos Hitam", 75000, 50), ("Kaos Polos Putih", 75000, 40),
                               ("Jaket Hoodie", 250000, 15)]:
        erp_con.execute("INSERT INTO products VALUES (?, ?, ?, ?)", [str(uuid.uuid4()), name, price, stock])

if erp_con.execute("SELECT COUNT(*) FROM customers").fetchone()[0] == 0:
    erp_con.execute("INSERT INTO customers VALUES (?, ?, ?)", [str(uuid.uuid4()), "Budi Santoso", "budi@example.com"])

# seed beberapa order contoh agar ada data untuk laporan
if erp_con.execute("SELECT COUNT(*) FROM orders").fetchone()[0] == 0:
    products = erp_con.execute("SELECT id, price FROM products").fetchall()
    customer_id = erp_con.execute("SELECT id FROM customers LIMIT 1").fetchone()[0]
    import random
    for _ in range(20):
        order_id = str(uuid.uuid4())
        erp_con.execute("INSERT INTO orders VALUES (?, ?, 'completed', current_timestamp - INTERVAL (random()*7)::INT DAY)",
                        [order_id, customer_id])
        pid, price = random.choice(products)
        qty = random.randint(1, 5)
        erp_con.execute("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)",
                        [str(uuid.uuid4()), order_id, pid, qty, price])
print("Total order:", erp_con.execute("SELECT COUNT(*) FROM orders").fetchone()[0])

## 3. Query Data Ringkasan Penjualan (SQL Only)

In [ ]:
def get_sales_summary(period_days=7):
    rows = erp_con.execute("""
        SELECT p.name, SUM(oi.qty) AS total_qty, SUM(oi.qty*oi.price) AS revenue
        FROM order_items oi
        JOIN products p ON p.id = oi.product_id
        JOIN orders o ON o.id = oi.order_id
        WHERE o.created_at >= current_date - INTERVAL (?) DAY
        GROUP BY p.name ORDER BY revenue DESC
    """, [period_days]).fetchall()
    return rows

get_sales_summary(7)

## 4. Generate Narasi Laporan dengan Qwen
Karena Qwen lokal (apalagi varian kecil 0.5B) cenderung kurang natural dibanding model besar, prompt perlu instruksi format yang lebih eksplisit (jumlah kalimat, larangan mengulang data mentah).

In [ ]:
def generate_sales_report(period_days=7):
    rows = get_sales_summary(period_days)
    data_summary = "\n".join([f"- {r[0]}: {r[1]} unit terjual, revenue Rp{r[2]:,.0f}" for r in rows])

    prompt = f"""Kamu adalah analis bisnis. Berikut data penjualan {period_days} hari terakhir:
{data_summary}

Tulis ringkasan eksekutif TEPAT 3-4 kalimat dalam Bahasa Indonesia formal, mencakup:
produk terlaris, tren yang terlihat, dan satu rekomendasi bisnis yang actionable.
Jangan mengulang data mentah, langsung ke insight.
Ringkasan:"""

    return qwen_llm_call(prompt, stop=["\n\n"])

print(generate_sales_report(7))

### 🔨 TODO 1
Buat fungsi `generate_stock_alert()` yang mengambil produk dengan `stock < 10`, lalu meminta Qwen menulis narasi peringatan stok kritis beserta rekomendasi restock.

In [ ]:
# TODO: implementasikan generate_stock_alert
def generate_stock_alert(threshold=10):
    low_stock = erp_con.execute("SELECT name, stock FROM products WHERE stock < ?", [threshold]).fetchall()
    # TODO: susun prompt dan panggil qwen_llm_call(...)
    pass

**Solusi contoh:**
```python
def generate_stock_alert(threshold=10):
    low_stock = erp_con.execute("SELECT name, stock FROM products WHERE stock < ?", [threshold]).fetchall()
    if not low_stock:
        return "Tidak ada produk dengan stok kritis saat ini."
    summary = "\n".join([f"- {n}: sisa {s} unit" for n, s in low_stock])
    prompt = f"Tulis 2-3 kalimat peringatan stok kritis dan rekomendasi restock untuk produk berikut:\n{summary}\nPeringatan:"
    return qwen_llm_call(prompt, stop=["\n\n"])
```

### 🔨 TODO 2 — Perbandingan
Bandingkan laporan SQL-only (`get_sales_summary`) vs narasi Qwen (`generate_sales_report`). Diskusikan: kapan laporan mentah lebih tepat, kapan narasi lebih membantu (misal untuk audiens non-teknis)? Amati juga kualitas bahasa Qwen — apakah perlu prompt lebih ketat atau few-shot example?

In [ ]:
print("=== SQL Only ===")
print(get_sales_summary(7))
print("\n=== Narasi Qwen ===")
print(generate_sales_report(7))

## Ringkasan Sesi 7
Data ERP mentah kini bisa diubah jadi laporan naratif otomatis via Qwen lokal — sepenuhnya on-premise, tanpa data bisnis terkirim ke API eksternal. Sesi 8 menggabungkan seluruh komponen (Knowledge Agent + Knowledge ERP) menjadi satu orchestrator.